
# GPT (Decoder) based distributed training

70p-30np Distribution



## Total Object Models: 9


## Training

1.	Camping
2.	Customer_Order
3.	Ecommerce
4.	Onlinestore
5.	Decider
6.  Library OM
7.  CSOS
8.  Flagship

## Testing

9.	Bank





--------------------------------

## Total Training Data: 13236 (100%)
--------------------------------

### Training set P :  9265 (70% of Training Data)

### Training set NP : 3971 (30% of Training Data)

-----------------------------
## Total Testing Data: 32 (100% of Total Data)
----------------------------

### Testing set P : 10 (31% of Testing Data)

### Testing set NP : 22 (69%% of Testing Data)

In [2]:
!pip install openai pandas numpy python-dotenv tqdm scikit-learn

In [3]:
! pip install openai==0.28

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.5/76.5 kB 7.2 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 1.54.5
    Uninstalling openai-1.54.5:
      Successfully uninstalled openai-1.54.5


In [4]:
!pip install openai==0.28


In [5]:
# Step 1: Import necessary libraries
import pandas as pd
import numpy as np
import openai
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
import time
from tqdm.notebook import tqdm


In [14]:
# Step 2: Set up OpenAI API key
openai.api_key = ""


In [22]:
# Step 3: Define the classification function
def classify_text_gpt4(text):
    try:
        response = openai.ChatCompletion.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": "You are a classifier that responds with exactly 'P' or 'NP'. Nothing else."},
                {"role": "user", "content": f"Classify the following text as either 'P' or 'NP': {text}"}
            ],
            temperature=0,
            max_tokens=1
        )
        return response.choices[0].message["content"].strip()
    except Exception as e:
        print(f"Error in classification: {e}")
        return None


In [23]:
# Step 4: Load training data
def load_training_data(file_path):
    df = pd.read_csv(file_path)
    return df['OM_Regular'].tolist(), df['OM_Prediction'].tolist()


In [24]:
# Step 5: Load test data
def load_test_data(file_path):
    df = pd.read_excel(file_path)
    return df['OM_Regular'].values, df['OM_Prediction'].values


In [25]:
# Step 6: Function to process a batch of texts with rate limiting
def process_batch_with_rate_limit(texts, batch_size=10):
    predictions = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        batch_predictions = []

        for text in batch:
            prediction = classify_text_gpt4(text)
            batch_predictions.append(prediction)
            time.sleep(1)  # Rate limiting to avoid hitting API limits

        predictions.extend(batch_predictions)
    return predictions


In [26]:
# Step 7: Calculate and display metrics
def calculate_metrics(y_true, y_pred):
    # Ensure both true and predicted labels are strings and filter out invalid predictions
    valid_indices = [i for i in range(len(y_pred)) if y_pred[i] is not None]
    y_true = [str(y_true[i]) for i in valid_indices]
    y_pred = [str(y_pred[i]) for i in valid_indices]

    precision = precision_score(y_true, y_pred, pos_label='P', zero_division=0)
    recall = recall_score(y_true, y_pred, pos_label='P', zero_division=0)
    f1 = f1_score(y_true, y_pred, pos_label='P', zero_division=0)

    print("\nMetrics:")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))


In [27]:
# Step 8: Save results
def save_results(texts, true_labels, predicted_labels, output_file):
    results_df = pd.DataFrame({
        'Text': texts,
        'True_Label': true_labels,
        'Predicted_Label': predicted_labels
    })
    results_df.to_csv(output_file, index=False)
    print(f"\nResults saved to {output_file}")


In [28]:
# Main execution
if __name__ == "__main__":
    try:
        # Test API connection
        test_response = classify_text_gpt4("This is a test text")
        if test_response:
            print("API Connection Test Successful!")
        else:
            raise Exception("API Test failed.")

        # Load training data
        print("\nLoading training data...")
        training_file = 'raw_8_om_training_set.csv'  # Update with your training file path
        X_train, y_train = load_training_data(training_file)

        print(f"Training Data Loaded: {len(X_train)} samples.")

        # Split the data into training and validation sets
        X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
            X_train, y_train, test_size=0.2, random_state=42
        )
        print(f"Training Data Split: {len(X_train_split)} training, {len(X_val_split)} validation samples.")

        # Validate the model on the validation set
        print("\nValidating on validation set...")
        y_val_pred = process_batch_with_rate_limit(X_val_split)
        calculate_metrics(y_val_split, y_val_pred)

        # Load test data
        print("\nLoading test data...")
        test_file = 'raw_testset_bank.xlsx'  # Update with your test file path
        X_test, y_test = load_test_data(test_file)

        # Make predictions on test data
        print("\nMaking predictions on test data...")
        y_pred = process_batch_with_rate_limit(X_test)

        # Calculate metrics for test data
        print("\nCalculating metrics on test data...")
        calculate_metrics(y_test, y_pred)

        # Save results
        save_results(X_test, y_test, y_pred, 'gpt4_classification_results.csv')

    except Exception as e:
        print(f"An error occurred: {e}")


API Connection Test Successful!

Loading training data...
Training Data Loaded: 13336 samples.
Training Data Split: 10668 training, 2668 validation samples.

Validating on validation set...


  0%|          | 0/267 [00:00<?, ?it/s]


Metrics:
Precision: 0.6959
Recall: 0.6930
F1 Score: 0.6945

Confusion Matrix:
[[ 194  575]
 [ 583 1316]]

Classification Report:
              precision    recall  f1-score   support

          NP       0.25      0.25      0.25       769
           P       0.70      0.69      0.69      1899

    accuracy                           0.57      2668
   macro avg       0.47      0.47      0.47      2668
weighted avg       0.57      0.57      0.57      2668


Loading test data...

Making predictions on test data...


  0%|          | 0/4 [00:00<?, ?it/s]


Calculating metrics on test data...
An error occurred: Target is multiclass but average='binary'. Please choose another average setting, one of [None, 'micro', 'macro', 'weighted'].


# Correction for making predictions from the Testset

In [50]:
# Import necessary libraries
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the test data
def load_test_data(file_path):
    df = pd.read_excel(file_path)
    return df['OM_Regular'].values, df['OM_Prediction'].values

# Function to calculate and display metrics
def calculate_metrics(y_true, y_pred):
    try:
        # Use 'micro', 'macro', or 'weighted' average for multiclass tasks
        precision = precision_score(y_true, y_pred, average='weighted')  # Can change to 'macro' or 'micro'
        recall = recall_score(y_true, y_pred, average='weighted')        # Same here
        f1 = f1_score(y_true, y_pred, average='weighted')                # Same here

        print("\nMetrics:")
        print(f"Precision: {precision:.4f}")
        print(f"Recall: {recall:.4f}")
        print(f"F1 Score: {f1:.4f}")

        print("\nConfusion Matrix:")
        print(confusion_matrix(y_true, y_pred))

        print("\nClassification Report:")
        print(classification_report(y_true, y_pred))

    except Exception as e:
        print(f"Error in calculating metrics: {e}")

# Main Execution Block for Testing Data Evaluation
if __name__ == "__main__":
    try:
        # Loading the test data from the file
        print("\nLoading test data...")
        test_file = 'raw_testset_bank.xlsx'  # Update with your test file path
        X_test, y_test = load_test_data(test_file)

        # Print sample size
        print(f"Test Data Loaded: {len(X_test)} samples.")

        # Making predictions on test data (Assume you have the model inference function)
        print("\nMaking predictions on test data...")
        y_pred = process_batch_with_rate_limit(X_test)  # You would replace this with actual prediction function

        # Calculate and display metrics for the test data
        print("\nCalculating metrics on test data...")
        calculate_metrics(y_test, y_pred)

        # Optional: Save results to CSV if needed
        save_results(X_test, y_test, y_pred, 'raw_testset_bank_pred_2.csv')

    except Exception as e:
        print(f"An error occurred: {e}")



Loading test data...
Test Data Loaded: 32 samples.

Making predictions on test data...


  0%|          | 0/4 [00:00<?, ?it/s]


Calculating metrics on test data...
Error in calculating metrics: Mix of label input types (string and number)

Results saved to raw_testset_bank_pred_2.csv


# Calculating Results from unseen Testset


In [51]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, cross_val_predict
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, roc_curve, roc_auc_score
from sklearn.metrics import precision_recall_curve, classification_report

In [52]:
dc = pd.read_excel('raw_testset_bank.xlsx')

In [53]:
X_test2 = dc['OM_Regular'].values
y_test2 = dc['OM_Prediction'].values

In [54]:
print(X_test2.shape)
print(y_test2.shape)

print("X data type: ", X_test2.dtype)
print("y data type: ", y_test2.dtype)

(32,)
(32,)
X data type:  object
y data type:  int64


In [55]:
print(y_test2)

[1 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 1 1 1 1 1]


In [56]:
dd = pd.read_excel('raw_testset_bank_pred_2.xlsx')

In [57]:
X_test_pred2 = dd['OM_Regular'].values
y_test_pred2 = dd['OM_Prediction'].values

In [58]:
print (y_test_pred2 )

[0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 0]


In [59]:
precision = precision_score(y_test2, y_test_pred2)
print("Testing: Precision = %f" % precision)


recall = recall_score(y_test2, y_test_pred2)
print("Testing: Recall = %f" % recall)


f1 = f1_score(y_test2, y_test_pred2)
print("Testing: F1 Score = %f" % f1)

print("\nConfusion Matrix (Test Data):\n", confusion_matrix(y_test2, y_test_pred2))

Testing: Precision = 0.250000
Testing: Recall = 0.100000
Testing: F1 Score = 0.142857

Confusion Matrix (Test Data):
 [[19  3]
 [ 9  1]]


In [60]:
print(classification_report(y_test2,y_test_pred2))

              precision    recall  f1-score   support

           0       0.68      0.86      0.76        22
           1       0.25      0.10      0.14        10

    accuracy                           0.62        32
   macro avg       0.46      0.48      0.45        32
weighted avg       0.54      0.62      0.57        32

